In [1]:
cd ..

c:\Users\Lenovo\Desktop\FYP\Tourist-Attraction-Recommendation-System


c:\Users\Lenovo\Desktop\FYP\Tourist-Attraction-Recommendation-System\venv\Lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


# Import Packages

In [18]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

import random
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import matplotlib.pyplot as plt
%matplotlib inline



# Hyperparameters

In [3]:
lr = 0.001
batch_size = 64
num_layers = 5
latent_dim = 8
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

# Dataset Preparation

In [4]:
from sklearn.preprocessing import LabelEncoder
import torch
from torch.utils.data import Dataset

class NCFDataset(Dataset):
    def __init__(self, user, item, rating):
        super(Dataset, self).__init__()
        
        # Encode user_id and place_id
        self.user_encoder = LabelEncoder()
        self.item_encoder = LabelEncoder()
        
        self.user = torch.LongTensor(self.user_encoder.fit_transform(user))
        self.item = torch.LongTensor(self.item_encoder.fit_transform(item))
        self.rating = torch.FloatTensor(rating.values)
        
        self.num_users = len(self.user_encoder.classes_)
        self.num_items = len(self.item_encoder.classes_)
        

    
    def __len__(self):
        return len(self.user)
    
    def __getitem__(self, idx):
        user = self.user[idx]
        item = self.item[idx]
        rating = self.rating[idx]
        
        return user, item, rating


In [5]:
# Load datasets
attraction_df = pd.read_csv('./Data/FinalDataset/Data.csv')
ratings_df = pd.read_csv('./Data/FinalDataset/rating_final.csv')
ratings_df.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3074 entries, 0 to 3073
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   user_id   3074 non-null   int64
 1   place_id  3074 non-null   int64
 2   rating    3074 non-null   int64
dtypes: int64(3)
memory usage: 72.2 KB


In [6]:

# Extract raw data
user = ratings_df['user_id']
item = ratings_df['place_id']
rating = ratings_df['rating']

# Train-test split
user_train, user_test, item_train, item_test, rating_train, rating_test = train_test_split(
    user, item, rating, test_size=0.2, random_state=42
)

# Create datasets
train_dataset = NCFDataset(user_train, item_train, rating_train)
test_dataset = NCFDataset(user_test, item_test, rating_test)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


# Model Building

In [7]:
class GMF(nn.Module):
    def __init__(self, num_users, num_items, latent_dim):
        super(GMF, self).__init__()
        self.user_embedding = nn.Embedding(num_users, latent_dim)   
        self.item_embedding = nn.Embedding(num_items, latent_dim)  
        nn.init.xavier_uniform_(self.user_embedding.weight)
        nn.init.xavier_uniform_(self.item_embedding.weight) 
    
    def forward(self, user, item):
        user_embedding = self.user_embedding(user)
        item_embedding = self.item_embedding(item)
        return user_embedding * item_embedding


In [8]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_layers, dropout_rate):
        super(MLP, self).__init__()
        layers = []
        for i, units in enumerate(hidden_layers):
            layers.extend([
                nn.Linear(input_dim, units),
                nn.BatchNorm1d(units),
                nn.ReLU(),
                nn.Dropout(p=dropout_rate)
            ])
            input_dim = units
        self.mlp = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self):
        for m in self.mlp:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.mlp(x)
        

In [12]:
class NCF(nn.Module):
    def __init__(self, num_users, num_items, latent_dim=64, hidden_layers=[256, 128, 64], dropout_rate=0.2):
        super(NCF, self).__init__()
        
        self.gmf = GMF(num_users, num_items, latent_dim)
        self.mlp = MLP(2 * latent_dim, hidden_layers, dropout_rate)
        
        # Separate embeddings for MLP
        self.mlp_user_embedding = nn.Embedding(num_users, latent_dim)
        self.mlp_item_embedding = nn.Embedding(num_items, latent_dim)
        
        fusion_dim = latent_dim + hidden_layers[-1]
        self.output_layer = nn.Sequential(
            nn.Linear(fusion_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(32, 1),
            nn.Sigmoid()  # Sigmoid to map to 0-1 range
        )
        

    def forward(self, user_ids, item_ids):
        # GMF path
        gmf_output = self.gmf(user_ids, item_ids)
        
        # MLP path with concatenated embeddings
        mlp_user_emb = self.mlp_user_embedding(user_ids)
        mlp_item_emb = self.mlp_item_embedding(item_ids)
        mlp_input = torch.cat([mlp_user_emb, mlp_item_emb], dim=-1)
        mlp_output = self.mlp(mlp_input)
        
        combined = torch.cat([gmf_output, mlp_output], dim=-1)
        output = self.output_layer(combined)
        
        # Scale output to 1-5 range, with 0 representing no rating
        rating = output * 5
        return rating

    def get_top_n_recommendations(self, user_ids, all_items, n=10):
        """
        Generate top-N recommendations for given users
        
        Args:
        - user_ids (torch.Tensor): Tensor of user IDs to generate recommendations for
        - all_items (torch.Tensor): Tensor of all possible item IDs
        - n (int): Number of top recommendations to return
        
        Returns:
        - Tensor of top-N item recommendations for each user
        """
        with torch.no_grad():
            # Create a matrix of predictions for each user-item combination
            user_expanded = user_ids.unsqueeze(1).expand(-1, len(all_items))
            item_expanded = all_items.unsqueeze(0).expand(len(user_ids), -1)
            
            predictions = self(user_expanded.flatten(), item_expanded.flatten())
            predictions = predictions.view(len(user_ids), -1)
            
            # Get top-N items for each user
            top_n_items = torch.topk(predictions, n, dim=1).indices
            
            return top_n_items

In [10]:
def recommend_top_n(model, user_ids, all_items, n=10):
    """
    Wrapper function to get top-N recommendations
    
    Args:
    - model (NCF): Trained NCF model
    - user_ids (list or torch.Tensor): User IDs to generate recommendations for
    - all_items (list or torch.Tensor): All possible item IDs
    - n (int): Number of top recommendations to return
    
    Returns:
    - DataFrame with user_id and recommended items
    """
    # Convert inputs to tensor if they're not already
    user_ids = torch.tensor(user_ids) if not isinstance(user_ids, torch.Tensor) else user_ids
    all_items = torch.tensor(all_items) if not isinstance(all_items, torch.Tensor) else all_items
    
    # Move tensors to the same device as the model
    user_ids = user_ids.to(next(model.parameters()).device)
    all_items = all_items.to(next(model.parameters()).device)
    
    # Get recommendations
    top_n_recommendations = model.get_top_n_recommendations(user_ids, all_items, n)
    
    # Convert back to numpy for easier handling
    top_n_recommendations = top_n_recommendations.cpu().numpy()
    
    # Create DataFrame with recommendations
    recommendations_df = pd.DataFrame({
        'user_id': np.repeat(user_ids.cpu().numpy(), n),
        'recommended_item': top_n_recommendations.ravel()
    })
    
    return recommendations_df

# Model Training

In [13]:
# Initialize the model
hidden_layers = [2**(7 - i) for i in range(num_layers)]

model = NCF(
    num_users=train_dataset.num_users, 
    num_items=train_dataset.num_items, 
    latent_dim=latent_dim, 
    hidden_layers=hidden_layers
).to(device)

# Loss function
criterion = nn.MSELoss()

# Optimizer with learning rate and optional weight decay
learning_rate = 0.001
optimizer = optim.Adam(
    model.parameters(), 
    lr=learning_rate, 
    weight_decay=1e-5  # Optional regularization
)

In [14]:
def evaluate_topn(model, test_loader, num_items, top_k=10):
    """
    Calculate evaluation metrics for top-N recommendations
    
    Args:
    - model: Trained NCF model
    - test_loader: DataLoader for test set
    - num_items: Total number of items
    - top_k: Number of top recommendations to consider
    
    Returns:
    - hit_rate, ndcg, rmse, precision, recall, mae
    """
    model.eval()
    device = next(model.parameters()).device
    
    # Containers for metrics calculation
    all_predictions = []
    all_true_ratings = []
    all_user_items = []
    hit_count = 0
    total_users = 0
    
    # Prepare all items for recommendation
    all_items = torch.arange(num_items).to(device)
    
    with torch.no_grad():
        for batch in test_loader:
            user_ids, item_ids, ratings = [b.to(device) for b in batch[:3]]
            
            # Filter out zero ratings
            mask = ratings > 0
            user_ids_valid = user_ids[mask]
            item_ids_valid = item_ids[mask]
            ratings_valid = ratings[mask]
            
            # Predictions for valid items
            predictions = model(user_ids_valid, item_ids_valid).squeeze()
            
            # Store for metric calculations
            all_predictions.extend(predictions.cpu().numpy())
            all_true_ratings.extend(ratings_valid.cpu().numpy())
            
            # Top-N recommendations for unique users
            unique_users = torch.unique(user_ids_valid)
            top_n_recommendations = model.get_top_n_recommendations(
                unique_users, all_items, n=top_k
            )
            
            # Hit Rate Calculation
            for user, rec_items in zip(unique_users, top_n_recommendations):
                # Find the ground truth items for this user
                user_true_items = item_ids_valid[user_ids_valid == user]
                
                # Check if any true item is in top-k recommendations
                if any(true_item in rec_items for true_item in user_true_items):
                    hit_count += 1
                total_users += 1
    
    # Metric Calculations
    hit_rate = hit_count / total_users if total_users > 0 else 0
    
    # RMSE
    rmse = np.sqrt(np.mean(
        [(pred - true)**2 for pred, true in zip(all_predictions, all_true_ratings)]
    ))
    
    # MAE
    mae = np.mean(
        [abs(pred - true) for pred, true in zip(all_predictions, all_true_ratings)]
    )
    
    # NDCG (simplified version)
    def ndcg_score(recommendations, ground_truth):
        # Check if ground truth is in recommendations
        if ground_truth in recommendations:
            rank = list(recommendations).index(ground_truth)
            return 1 / np.log2(rank + 2)
        return 0
    
    ndcg = np.mean([
        ndcg_score(top_n_recommendations[i], item_ids_valid[i]) 
        for i in range(len(top_n_recommendations))
    ])
    
    # Precision and Recall
    precision = hit_count / (total_users * top_k) if total_users > 0 else 0
    recall = hit_count / total_users if total_users > 0 else 0
    
    return hit_rate, ndcg, rmse, precision, recall, mae

In [15]:
def training_loop(model, train_loader, test_loader, optimizer, criterion, num_epochs, device, num_items, top_k=10):
    """
    Comprehensive training loop with metrics tracking
    
    Args:
    - model: NCF model
    - train_loader: DataLoader for training
    - test_loader: DataLoader for testing
    - optimizer: Optimizer
    - criterion: Loss function
    - num_epochs: Number of training epochs
    - device: Training device
    - num_items: Total number of items
    - top_k: Number of top recommendations
    
    Returns:
    - Dictionaries of tracked metrics
    """
    # Metric tracking lists
    train_losses = []
    test_losses = []
    hit_rates = []
    ndcgs = []
    rmses = []
    precisions = []
    recalls = []
    maes = []
    
    # Best metrics tracking
    best_hit_rate = 0.0
    best_ndcg = 0.0
    best_rmse = float('inf')
    best_precision = 0.0
    best_recall = 0.0
    best_mae = float('inf')
    best_epoch = -1
    
    # Training loop
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        
        for user_ids, item_ids, ratings in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
            user_ids, item_ids, ratings = user_ids.to(device), item_ids.to(device), ratings.to(device)
            
            optimizer.zero_grad()
            predictions = model(user_ids, item_ids)
            loss = criterion(predictions.squeeze(), ratings)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        avg_train_loss = running_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        
        # Evaluation phase
        model.eval()
        with torch.no_grad():
            test_loss = 0.0
            for user_ids, item_ids, ratings in test_loader:
                user_ids, item_ids, ratings = user_ids.to(device), item_ids.to(device), ratings.to(device)
                predictions = model(user_ids, item_ids)
                loss = criterion(predictions.squeeze(), ratings)
                test_loss += loss.item()
            
            avg_test_loss = test_loss / len(test_loader)
            test_losses.append(avg_test_loss)
        
        # Calculate evaluation metrics
        hit_rate, ndcg, rmse, precision, recall, mae = evaluate_topn(
            model, test_loader, num_items, top_k
        )
        
        # Store metrics
        hit_rates.append(hit_rate)
        ndcgs.append(ndcg)
        rmses.append(rmse)
        precisions.append(precision)
        recalls.append(recall)
        maes.append(mae)
        
        # Print epoch metrics
        print(f"Epoch {epoch+1}/{num_epochs}, "
              f"Training Loss: {avg_train_loss:.4f}, "
              f"Test Loss: {avg_test_loss:.4f}, "
              f"Hit Rate@{top_k}: {hit_rate:.4f}, "
              f"NDCG@{top_k}: {ndcg:.4f}, "
              f"RMSE: {rmse:.4f}, "
              f"MAE: {mae:.4f}, "
              f"Precision@{top_k}: {precision:.4f}, "
              f"Recall@{top_k}: {recall:.4f}")
        
        # Update best metrics
        if hit_rate > best_hit_rate:
            best_hit_rate = hit_rate
            best_ndcg = ndcg
            best_rmse = rmse
            best_mae = mae
            best_precision = precision
            best_recall = recall
            best_epoch = epoch
        elif hit_rate == best_hit_rate and ndcg > best_ndcg:
            best_ndcg = ndcg
            best_rmse = rmse
            best_mae = mae
            best_precision = precision
            best_recall = recall
            best_epoch = epoch
    
    # Print best metrics
    print("\n\nBest Epoch:", best_epoch + 1)
    print(f"Hit Rate@{top_k}: {best_hit_rate:.4f}")
    print(f"NDCG@{top_k}: {best_ndcg:.4f}")
    print(f"RMSE: {best_rmse:.4f}")
    print(f"MAE: {best_mae:.4f}")
    print(f"Precision@{top_k}: {best_precision:.4f}")
    print(f"Recall@{top_k}: {best_recall:.4f}")
    
    # Return tracking dictionaries for further analysis
    return {
        'train_losses': train_losses,
        'test_losses': test_losses,
        'hit_rates': hit_rates,
        'ndcgs': ndcgs,
        'rmses': rmses,
        'maes': maes,
        'precisions': precisions,
        'recalls': recalls
    }

In [16]:
training_loop(
    model, train_loader, test_loader, optimizer, criterion, num_epochs=10, 
    device=device, num_items=train_dataset.num_items, top_k=10
)

Epoch 1/10: 100%|██████████| 77/77 [00:00<00:00, 113.37it/s]


Epoch 1/10, Training Loss: 2.7818, Test Loss: 2.0456, Hit Rate@10: 0.0131, NDCG@10: 0.0553, RMSE: 1.4216, MAE: 1.3158, Precision@10: 0.0013, Recall@10: 0.0131


Epoch 2/10: 100%|██████████| 77/77 [00:00<00:00, 186.04it/s]


Epoch 2/10, Training Loss: 1.2493, Test Loss: 1.1019, Hit Rate@10: 0.0278, NDCG@10: 0.0000, RMSE: 1.0369, MAE: 0.8481, Precision@10: 0.0028, Recall@10: 0.0278


Epoch 3/10: 100%|██████████| 77/77 [00:00<00:00, 162.55it/s]


...

Epoch 9/10: 100%|██████████| 77/77 [00:00<00:00, 176.11it/s]


Epoch 9/10, Training Loss: 0.5418, Test Loss: 1.0962, Hit Rate@10: 0.0245, NDCG@10: 0.0000, RMSE: 1.0337, MAE: 0.8263, Precision@10: 0.0025, Recall@10: 0.0245


Epoch 10/10: 100%|██████████| 77/77 [00:00<00:00, 180.28it/s]


Epoch 10/10, Training Loss: 0.3755, Test Loss: 1.1427, Hit Rate@10: 0.0262, NDCG@10: 0.0000, RMSE: 1.0568, MAE: 0.8456, Precision@10: 0.0026, Recall@10: 0.0262


Best Epoch: 2
Hit Rate@10: 0.0278
NDCG@10: 0.0000
RMSE: 1.0369
MAE: 0.8481
Precision@10: 0.0028
Recall@10: 0.0278


{'train_losses': [2.781844735145569,
  1.249299221224599,
  1.0151231226983009,
  0.9823053514028525,
  0.9755341820902639,
  0.9323098965279468,
  0.8517272385296883,
  0.7128101215734111,
  0.541823411723236,
  0.37546088459429805],
 'test_losses': [2.0456336438655853,
  1.1019118070602416,
  1.0298606976866722,
  1.0092387832701206,
  1.0126252889633178,
  1.0177320495247841,
  1.0501392036676407,
  1.0625506401062013,
  1.0962425246834755,
  1.14269260764122],
 'hit_rates': [0.01309328968903437,
  0.027823240589198037,
  0.01800327332242226,
  0.01800327332242226,
  0.02618657937806874,
  0.02618657937806874,
  0.014729950900163666,
  0.02127659574468085,
  0.024549918166939442,
  0.02618657937806874],
 'ndcgs': [np.float64(0.05526468674779166),
  np.float64(0.0),
  np.float64(0.0),
  np.float64(0.0),
  np.float64(0.0),
  np.float64(0.0),
  np.float64(0.043004285094854454),
  np.float64(0.04506641096938983),
  np.float64(0.0),
  np.float64(0.0)],
 'rmses': [np.float32(1.4216465),
 

In [19]:
# Plotting the training and test loss curves
plt.figure(figsize=(10, 6))
plt.plot(range(1, num_epochs+1), train_losses, label="Training Loss", color="blue")
plt.plot(range(1, num_epochs+1), test_losses, label="Test Loss", color="red")
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and Test Loss Curve')
plt.legend()
plt.grid(True)
plt.show()

NameError: name 'num_epochs' is not defined

<Figure size 1000x600 with 0 Axes>